# Notebook 04: Baseline decision tree

**Project:** Lightweight machine-learning-based cybersecurity threat detection for resource-constrained hardware
**Author:** Anush Jindal

Trains an untuned decision tree on each of the four feature lists A, B, C, D from
notebook 03 and measures both detection (accuracy, macro F1, attack recall, missed attacks,
false alarms) and cost (nodes, depth, saved size, training and prediction time, memory).
This is the **baseline** that the smaller trees in notebook 05 are compared against.

## Rules

1. **The tree learns from `X_train` only.** `X_test` is used only for scoring, never to
   choose features or settings.
2. **No tuning.** `random_state=42` and every other setting at its default: no
   `max_depth`, no pruning. Shrinking the tree is notebook 05.

## Steps

1. Setup
2. Load the data and the four feature lists
3. Test the keep/remove claims from the feature table
4. Train on list A and score it
5. Compare A, B, C, D
6. Shortcut check
7. Save the results
8. Caveats
9. Follow-up checks: scores without Heartbleed, and memory use

## Where the results are

The final Stage 5 numbers come from the re-run of Steps 4-9 in notebook 05, Part 1
(29-10-2026, with the 58-feature list A), and are in `experiments/results.csv` and
`experiments/S5_*.csv`. This notebook's own outputs are not saved (see the change log).

## Change log

- **23-10-2026:** `Fwd PSH Flags` was added back to list A (A is now **58** features, was 57).
  Step 3 showed it is not "always zero" (non-zero in 4.32% of rows). Lists B, C and D did
  not change. The checks below expect the new sizes, so notebook 03 must be re-run first
  to re-save `feature_sets.json` with the new list A.
- **23-10-2026:** Step 9 was run for the first time (with the old 57-feature list A).
- Saved outputs were cleared on 23-10-2026 because the code changed. The numbers from
  earlier runs are recorded in `experiments/experiment_notes.md`.

## Step 1 — Setup

Imports, and the Drive folder used by notebook 03: inputs are read from `data/`, trained
trees are saved to `models/` and result files to `experiments/`.

In [ ]:
# Step 1 — Setup
import os
import time
import json
import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
)

%matplotlib inline

# Mount Google Drive (only works inside Colab; skipped when run locally)
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Not running in Colab, so Drive was not mounted.")

# Same folder Notebook 3 saved into
PROJECT_DIR = Path("/content/drive/MyDrive/Colab Notebooks/threat detector")
DATA_DIR    = PROJECT_DIR / "data"          # inputs from Notebook 3
MODEL_DIR   = PROJECT_DIR / "models"        # saved trees (.joblib) go here
RESULTS_DIR = PROJECT_DIR / "experiments"   # results CSV + figures go here
MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Data folder:", DATA_DIR)
print("Files in it:", sorted(p.name for p in DATA_DIR.iterdir()))

## Step 2 — Load the data and the four feature lists

| File | Contents |
|---|---|
| `X_train_engineered.csv` | Features of the training set |
| `X_test_engineered.csv` | Features of the test set |
| `y_train.csv` | Labels of the training set |
| `y_test.csv` | Labels of the test set |
| `feature_sets.json` | Lists A, B, C, D |

**Checks** (the notebook stops if one fails):
- The shapes match notebook 02: (488393, 78) and (122099, 78).
- The four lists have **58, 39, 15 and 6** names (A was 57 before 23-10-2026, when `Fwd PSH Flags` was added back).
- Train and test have the same columns in the same order.
- Every name in every list exists in `X_train`.
- No `NaN` or `inf` values are left, and the label `BENIGN` exists exactly as spelled.

In [ ]:
# Step 2 — Load the data and the four feature lists
X_train = pd.read_csv(DATA_DIR / "X_train_engineered.csv")
X_test  = pd.read_csv(DATA_DIR / "X_test_engineered.csv")
# squeeze("columns") turns the one-column table into a plain Series of labels
y_train = pd.read_csv(DATA_DIR / "y_train.csv").squeeze("columns")
y_test  = pd.read_csv(DATA_DIR / "y_test.csv").squeeze("columns")

with open(DATA_DIR / "feature_sets.json") as f:
    feature_sets = json.load(f)

print("X_train:", X_train.shape, "  X_test:", X_test.shape)
print("y_train:", y_train.shape, "  y_test:", y_test.shape)
print("Feature list sizes:", {name: len(cols) for name, cols in feature_sets.items()})

# ---------------- Checks ----------------
# 1. Shapes match the notes from Notebook 2 (experiment_notes.md, 6-10-2026)
EXPECTED_TRAIN_SHAPE = (488393, 78)
EXPECTED_TEST_SHAPE  = (122099, 78)
assert X_train.shape == EXPECTED_TRAIN_SHAPE, f"X_train is {X_train.shape}, notes say {EXPECTED_TRAIN_SHAPE}"
assert X_test.shape == EXPECTED_TEST_SHAPE, f"X_test is {X_test.shape}, notes say {EXPECTED_TEST_SHAPE}"
assert isinstance(y_train, pd.Series) and isinstance(y_test, pd.Series), "y files did not squeeze to a Series"
assert len(y_train) == len(X_train) and len(y_test) == len(X_test), "X and y row counts differ"

# 2. The four lists have 58, 39, 15 and 6 names (and no name appears twice)
#    A was 57 until 23-10-2026, when Fwd PSH Flags was added back (see Step 3).
#    If this fails with "List A has 57 names", re-run Notebook 3 to re-save feature_sets.json.
EXPECTED_SIZES = {"A": 58, "B": 39, "C": 15, "D": 6}
assert "Fwd PSH Flags" in feature_sets["A"], "Fwd PSH Flags is missing from list A: re-run Notebook 3"
for name, n in EXPECTED_SIZES.items():
    assert len(feature_sets[name]) == n, f"List {name} has {len(feature_sets[name])} names, expected {n}"
    assert len(set(feature_sets[name])) == n, f"List {name} has a repeated name"

# 3. Train and test have the same columns in the same order
assert list(X_train.columns) == list(X_test.columns), "Train and test columns differ"

# 4. Every name in every list really exists in X_train
for name, cols in feature_sets.items():
    missing = [c for c in cols if c not in X_train.columns]
    assert not missing, f"List {name} has names not in X_train: {missing}"

# 5. Extra safety: all columns are numbers, with no NaN or inf left (a tree crashes on inf)
non_numeric = X_train.columns[~X_train.dtypes.apply(pd.api.types.is_numeric_dtype)].tolist()
assert not non_numeric, f"Non-numeric columns: {non_numeric}"
for nm, X in [("X_train", X_train), ("X_test", X_test)]:
    values = X.to_numpy(dtype=float)
    assert not np.isnan(values).any(), f"{nm} still has NaN values"
    assert not np.isinf(values).any(), f"{nm} still has inf values"

# 6. Label counts, and the exact text "BENIGN" is one of them
print("\nLabels in y_train:")
print(y_train.value_counts().to_string())
assert "BENIGN" in set(y_train.unique()), "No label is exactly 'BENIGN' (check spaces/case)"
assert set(y_test.unique()) <= set(y_train.unique()), "y_test has a label that y_train does not"

print("\nAll Step 2 checks passed.")

## Step 3 — Test the keep/remove claims

The feature table (`research/feature_documentation.md`) removes some columns from list A
because they are *identical* to a kept column, have the *same ordering* as one, or are
*always zero*. Those reasons came from the dataset literature, so they are checked here on
`X_train` (the test set is not used). If a claim does not hold, the table and the `A_drop`
list in notebook 03 are corrected.

**Part A — claimed identical.** Each pair is compared row by row with `np.allclose`.

| Dropped column | Compared with |
|---|---|
| Avg Fwd Segment Size | Fwd Packet Length Mean |
| Avg Bwd Segment Size | Bwd Packet Length Mean |
| Subflow Fwd Packets | Total Fwd Packets |
| Subflow Fwd Bytes | Total Length of Fwd Packets |
| Subflow Bwd Packets | Total Backward Packets |
| Subflow Bwd Bytes | Total Length of Bwd Packets |
| Fwd Header Length.1 | Fwd Header Length |
| ECE Flag Count | RST Flag Count |

**Part B — claimed same ordering.** Spearman correlation for *Packet Length Variance* vs
*Packet Length Std*, and *Average Packet Size* vs *Packet Length Mean*. A value of 1.0
means a tree gains nothing from having both, because a split only depends on the order
of the values.

**Part C — claimed always zero.** Number of distinct values and share of non-zero rows for
Bwd PSH Flags, Fwd URG Flags, Bwd URG Flags, CWE Flag Count and the six Bulk columns. One
distinct value means the column carries no information.

**Decisions (23-10-2026), after the first run of this step:**

| Column | What the data showed | Decision |
|---|---|---|
| Fwd PSH Flags | Non-zero in 4.32% of rows, so not "always zero" | **Added back to list A** (now 58 features) |
| Subflow Bwd Bytes | Differs from Total Length of Bwd Packets in only 34 of 488,393 rows | Still removed: nearly identical |
| ECE Flag Count | Differs from RST Flag Count in only 1 row | Still removed: nearly identical |
| Average Packet Size | Spearman 0.9976 with Packet Length Mean | Still removed: almost the same ordering |

Because `Fwd PSH Flags` is now kept, it is no longer in the Part C list. The other three
still show up as "did not hold" because they are *nearly* (not exactly) identical; the
cell labels them as already decided.

In [ ]:
# Step 3 — Test the keep/remove claims (on X_train only: the test set is not used)

# ---- Part A: claimed identical ----
identical_pairs = [
    ("Avg Fwd Segment Size", "Fwd Packet Length Mean"),
    ("Avg Bwd Segment Size", "Bwd Packet Length Mean"),
    ("Subflow Fwd Packets",  "Total Fwd Packets"),
    ("Subflow Fwd Bytes",    "Total Length of Fwd Packets"),
    ("Subflow Bwd Packets",  "Total Backward Packets"),
    ("Subflow Bwd Bytes",    "Total Length of Bwd Packets"),
    ("Fwd Header Length.1",  "Fwd Header Length"),
    ("ECE Flag Count",       "RST Flag Count"),
]
part_a_rows = []
for dropped, kept in identical_pairs:
    a = X_train[dropped].to_numpy(dtype=float)
    b = X_train[kept].to_numpy(dtype=float)
    part_a_rows.append({
        "dropped column": dropped,
        "compared with": kept,
        "identical": bool(np.allclose(a, b)),
        "rows that differ": int((~np.isclose(a, b)).sum()),
        "max abs difference": float(np.abs(a - b).max()),
    })
part_a = pd.DataFrame(part_a_rows)
print("Part A — claimed identical")
display(part_a)

# ---- Part B: claimed same ordering ----
rank_pairs = [
    ("Packet Length Variance", "Packet Length Std"),
    ("Average Packet Size",    "Packet Length Mean"),
]
part_b = pd.DataFrame([
    {"dropped column": x, "compared with": y,
     "spearman": X_train[x].corr(X_train[y], method="spearman")}
    for x, y in rank_pairs
])
print("Part B — claimed same ordering (1.0 = a tree gains nothing from having both)")
display(part_b)

# ---- Part C: claimed always zero (or nearly) ----
# (Fwd PSH Flags used to be in this list. It was added back to list A on 23-10-2026,
#  because this check found it non-zero in 4.32% of rows, so it is no longer a claim to test.)
zero_cols = [
    "Bwd PSH Flags", "Fwd URG Flags", "Bwd URG Flags", "CWE Flag Count",
    "Fwd Avg Bytes/Bulk", "Fwd Avg Packets/Bulk", "Fwd Avg Bulk Rate",
    "Bwd Avg Bytes/Bulk", "Bwd Avg Packets/Bulk", "Bwd Avg Bulk Rate",
]
part_c = pd.DataFrame({
    "distinct values": X_train[zero_cols].nunique(),
    "share of rows not zero": (X_train[zero_cols] != 0).mean(),
})
print("Part C — claimed always zero (1 distinct value = carries no information)")
display(part_c)

# ---- Which claims did not hold? ----
NEAR_ZERO_LIMIT = 0.01   # threshold for "nearly always zero": under 1% of rows non-zero
flags = []
for _, r in part_a.iterrows():
    if not r["identical"]:
        flags.append(f"{r['dropped column']} is NOT identical to {r['compared with']} "
                     f"({r['rows that differ']} rows differ)")
for _, r in part_b.iterrows():
    if r["spearman"] < 0.9999:
        flags.append(f"{r['dropped column']} vs {r['compared with']}: Spearman = {r['spearman']:.6f}, not exactly 1.0")
for col, r in part_c.iterrows():
    if r["share of rows not zero"] > NEAR_ZERO_LIMIT:
        flags.append(f"{col} is non-zero in {r['share of rows not zero']:.2%} of rows "
                     f"({int(r['distinct values'])} distinct values): more than the {NEAR_ZERO_LIMIT:.0%} limit for 'nearly always zero'")

# Claims already looked at and decided on 23-10-2026 (see the table above this cell)
ALREADY_DECIDED = {
    "Subflow Bwd Bytes":   "decided 23-10-2026: still removed, nearly identical",
    "ECE Flag Count":      "decided 23-10-2026: still removed, nearly identical",
    "Average Packet Size": "decided 23-10-2026: still removed, almost the same ordering",
}
print("\nClaims that did not hold:" if flags else "\nAll claims confirmed.")
for line in flags:
    col = line.split(" is NOT")[0].split(" vs ")[0].split(" is non-zero")[0]
    note = ALREADY_DECIDED.get(col, "NEW: not decided yet")
    print(" -", line, f"  [{note}]")

## Step 4 — Train on list A and score it

`run_experiment` trains one tree on a list of columns and returns how good and how
expensive it is. It is reused in Steps 5 and 6.

1. `DecisionTreeClassifier(random_state=42)`, no other settings.
2. Training time: `time.perf_counter()` around `fit`.
3. Prediction time on `X_test`: median of 5 runs, reported as milliseconds per 1,000 rows.
4. Measures:
   - accuracy, and macro precision, recall and F1 (`zero_division=0`)
   - **attack vs normal:** a row is an "attack" if its label is not `BENIGN`. **Caught** =
     real attack predicted as an attack, **missed** = real attack predicted as normal,
     **false alarm** = normal predicted as an attack. Attack recall = caught ÷ (caught + missed).
   - tree size: number of nodes and depth
   - saved model size: the `joblib` file size in KB
5. A per-class `classification_report`, and the confusion matrix as a heatmap with each
   row divided by its total, so rare attacks stay visible.

**Macro F1** averages all classes equally, so rare attacks count as much as common ones;
it is harsher than accuracy. **Attack recall** is the share of real attacks caught;
missing an attack is the costliest mistake.

In [ ]:
# Step 4 — Train on feature list A and score it
RANDOM_STATE = 42
BENIGN = "BENIGN"
N_PREDICT_REPEATS = 5          # predict() on the test set is fast, so one timing is noisy: take the median of 5
CLASS_LABELS = sorted(y_train.unique())

results = {}       # run name -> dictionary of numbers
predictions = {}   # run name -> predicted labels for X_test


def run_experiment(name, features):
    """Train an untuned tree on `features`, score it on the test set, save it, return the numbers."""
    features = list(features)
    Xtr, Xte = X_train[features], X_test[features]

    # 1. The baseline tree: random_state only, every other setting left at its default
    model = DecisionTreeClassifier(random_state=RANDOM_STATE)

    # 2. Time the training
    t0 = time.perf_counter()
    model.fit(Xtr, y_train)
    training_time_s = time.perf_counter() - t0

    # 3. Time the prediction on the test set
    pred_times = []
    for _ in range(N_PREDICT_REPEATS):
        t0 = time.perf_counter()
        y_pred = model.predict(Xte)
        pred_times.append(time.perf_counter() - t0)
    predict_s = float(np.median(pred_times))
    inference_ms_per_1000 = predict_s * 1000 / len(Xte) * 1000

    # 4a. Accuracy and macro precision / recall / F1
    accuracy = accuracy_score(y_test, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_test, y_pred, average="macro", zero_division=0)

    # 4b. Attack vs normal: "attack" = any label that is not BENIGN
    real_attack = (y_test != BENIGN).to_numpy()
    guessed_attack = (y_pred != BENIGN)
    caught       = int(( real_attack &  guessed_attack).sum())
    missed       = int(( real_attack & ~guessed_attack).sum())
    false_alarms = int((~real_attack &  guessed_attack).sum())
    attack_recall = caught / (caught + missed)

    # 4c. Tree size
    num_nodes = int(model.tree_.node_count)
    depth = int(model.get_depth())

    # 4d. Saved model size
    model_path = MODEL_DIR / f"S5_baseline_{name}.joblib"
    joblib.dump(model, model_path)
    model_size_kb = os.path.getsize(model_path) / 1024

    # Per-class recall (used in the Step 5/6 tables: which attacks get lost as the feature list shrinks?)
    _, class_recall, _, class_support = precision_recall_fscore_support(
        y_test, y_pred, labels=CLASS_LABELS, zero_division=0)

    res = {
        "run": name,
        "num_features": len(features),
        "accuracy": accuracy,
        "precision_macro": precision,
        "recall_macro": recall,
        "f1_macro": f1,
        "attack_recall": attack_recall,
        "caught": caught,
        "missed": missed,
        "false_alarms": false_alarms,
        "num_nodes": num_nodes,
        "depth": depth,
        "model_size_kb": model_size_kb,
        "training_time_s": training_time_s,
        "inference_ms_per_1000": inference_ms_per_1000,
        "per_class_recall": dict(zip(CLASS_LABELS, class_recall)),
        "per_class_support": dict(zip(CLASS_LABELS, class_support.astype(int))),
    }
    results[name] = res
    predictions[name] = y_pred

    print(f"[{name}] {len(features)} features | acc {accuracy:.4f} | macro F1 {f1:.4f} | "
          f"attack recall {attack_recall:.4f} (missed {missed}, false alarms {false_alarms}) | "
          f"{num_nodes} nodes, depth {depth}, {model_size_kb:.1f} KB | "
          f"train {training_time_s:.2f} s, predict {inference_ms_per_1000:.4f} ms/1000 rows")
    return res, model


res_A, model_A = run_experiment("A", feature_sets["A"])

# The results dictionary for list A (everything except the per-class detail)
{k: v for k, v in res_A.items() if not k.startswith("per_class")}

In [ ]:
# Step 4 (continued) — per-class report and confusion-matrix heatmap for list A
print(classification_report(y_test, predictions["A"], labels=CLASS_LABELS, digits=4, zero_division=0))

cm = confusion_matrix(y_test, predictions["A"], labels=CLASS_LABELS)
row_totals = cm.sum(axis=1, keepdims=True)
cm_share = np.divide(cm, row_totals, out=np.zeros(cm.shape), where=row_totals > 0)  # each row: share of that true class

fig, ax = plt.subplots(figsize=(8, 6.5))
sns.heatmap(cm_share, annot=True, fmt=".3f", cmap="Blues", vmin=0, vmax=1,
            xticklabels=CLASS_LABELS, yticklabels=CLASS_LABELS,
            linewidths=1, linecolor="white", cbar_kws={"label": "Share of the true class"}, ax=ax)
ax.set_xlabel("Predicted label")
ax.set_ylabel("True label")
ax.set_title("Untuned tree, feature list A: confusion matrix (each row sums to 1)")
plt.xticks(rotation=30, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
fig.savefig(RESULTS_DIR / "S5_A_confusion_matrix.png", dpi=150)
plt.show()

# Raw counts too: for rare classes a share like 0.500 might be just 1 row out of 2
print("Raw counts (rows = true, columns = predicted):")
display(pd.DataFrame(cm, index=CLASS_LABELS, columns=CLASS_LABELS))

# The class whose diagonal is furthest from 1.0, and what it is mistaken for
diag = np.diag(cm_share)
weakest = int(np.argmin(diag))
off_diag = cm_share[weakest].copy()
off_diag[weakest] = -1
confused_with = int(np.argmax(off_diag))
print(f"Lowest diagonal: {CLASS_LABELS[weakest]} = {diag[weakest]:.4f} "
      f"({int(cm[weakest, weakest])} of {int(row_totals[weakest, 0])} test rows correct). "
      f"Most often confused with {CLASS_LABELS[confused_with]} ({off_diag[confused_with]:.4f}).")

## Step 5 — Compare A, B, C, D

How much detection is lost with fewer features? The same function is run on lists B, C
and D. The four results go into one table, sorted from most to fewest features, with two
plots (macro F1 and saved model size against the number of features) and the recall of
each class. This is one run on one split; notebook 05 repeats it on five splits.

In [ ]:
# Step 5 — Compare A, B, C, D

for name in ["B", "C", "D"]:
    run_experiment(name, feature_sets[name])

TABLE_COLS = ["run", "num_features", "accuracy", "precision_macro", "recall_macro", "f1_macro",
              "attack_recall", "missed", "false_alarms", "num_nodes", "depth",
              "model_size_kb", "training_time_s", "inference_ms_per_1000"]


def results_table(names):
    return pd.DataFrame([results[n] for n in names])[TABLE_COLS]


compare_ABCD = results_table(["A", "B", "C", "D"]).sort_values("num_features", ascending=False)
display(compare_ABCD.round(4))

# Two small graphs, one measure each (separate charts: the two numbers have different units)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, col, ylabel in [(axes[0], "f1_macro", "Macro F1 (test)"),
                        (axes[1], "model_size_kb", "Saved model size (KB)")]:
    ax.plot(compare_ABCD["num_features"], compare_ABCD[col], marker="o", markersize=8,
            linewidth=2, color="#2a6fdb")
    for _, r in compare_ABCD.iterrows():
        ax.annotate(f"{r['run']}: {r[col]:.4f}" if col == "f1_macro" else f"{r['run']}: {r[col]:.0f}",
                    (r["num_features"], r[col]), textcoords="offset points", xytext=(6, 6), fontsize=9)
    ax.set_xlabel("Number of features")
    ax.set_ylabel(ylabel)
    ax.grid(alpha=0.3)
    ax.margins(x=0.15, y=0.15)   # room for the point labels
    ax.spines[["top", "right"]].set_visible(False)
axes[0].set_title("Macro F1 vs number of features")
axes[1].set_title("Model size vs number of features")
plt.tight_layout()
fig.savefig(RESULTS_DIR / "S5_ABCD_f1_and_size.png", dpi=150)
plt.show()


def recall_table(names):
    """Recall for each true class, one column per run; 'test rows' shows how much each number rests on."""
    t = pd.DataFrame({n: results[n]["per_class_recall"] for n in names})
    t.insert(0, "test rows", pd.Series(results[names[0]]["per_class_support"]))
    return t


print("Recall for each class (which attacks get lost as the feature list shrinks?):")
display(recall_table(["A", "B", "C", "D"]).round(4))

## Step 6 — Shortcut check

The CICIDS2017 attacks were run in a lab against fixed machines and ports. A tree might
score well by memorising *"this port or this TCP window size means attack"* instead of
learning how attacks behave, a shortcut that a real network would not offer. The feature
table flags `Destination Port` and the two `Init_Win_bytes` columns as possible shortcuts.

| Run name | Features |
|---|---|
| A_no_port | List A without `Destination Port` (57) |
| A_no_port_no_window | List A without `Destination Port`, `Init_Win_bytes_forward` and `Init_Win_bytes_backward` (55) |
| C_no_port | List C without `Destination Port` (14) |
| D_no_port | List D without `Destination Port` (5) |

If the score barely moves, the tree is learning behaviour; if it drops, it was leaning on
a shortcut. This asks a question about the model rather than tuning it: no list is
chosen by its test score.

In [ ]:
# Step 6 — Shortcut check: remove the lab-specific features and see if the score holds
PORT = "Destination Port"
WINDOW = ["Init_Win_bytes_forward", "Init_Win_bytes_backward"]


def without(cols, drop):
    return [c for c in cols if c not in drop]


shortcut_runs = {
    "A_no_port":           (without(feature_sets["A"], [PORT]),          "A"),
    "A_no_port_no_window": (without(feature_sets["A"], [PORT] + WINDOW), "A"),
    "C_no_port":           (without(feature_sets["C"], [PORT]),          "C"),
    "D_no_port":           (without(feature_sets["D"], [PORT]),          "D"),
}
# Make sure each removal actually happened (A: 58 -> 57 -> 55, C: 15 -> 14, D: 6 -> 5)
expected_counts = {"A_no_port": 57, "A_no_port_no_window": 55, "C_no_port": 14, "D_no_port": 5}
for name, (cols, _) in shortcut_runs.items():
    assert len(cols) == expected_counts[name], f"{name} has {len(cols)} features"

for name, (cols, _) in shortcut_runs.items():
    run_experiment(name, cols)

# Side by side with the run it came from
rows = []
for name, (_, parent) in shortcut_runs.items():
    r, p = results[name], results[parent]
    rows.append({
        "run": name, "compared with": parent,
        "macro F1": r["f1_macro"], "change in macro F1": r["f1_macro"] - p["f1_macro"],
        "attack recall": r["attack_recall"], "change in attack recall": r["attack_recall"] - p["attack_recall"],
        "missed": r["missed"], "change in missed": r["missed"] - p["missed"],
        "false alarms": r["false_alarms"], "change in false alarms": r["false_alarms"] - p["false_alarms"],
        "nodes": r["num_nodes"], "change in nodes": r["num_nodes"] - p["num_nodes"],
    })
shortcut_compare = pd.DataFrame(rows)
display(shortcut_compare.round(4))

print("Recall for each class, with and without the shortcut features:")
display(recall_table(["A", "A_no_port", "A_no_port_no_window", "C", "C_no_port", "D", "D_no_port"]).round(4))

## Step 7 — Save the results

The eight runs are written in the layout of `experiments/results.csv`:

`experiment_id, date, stage, dataset_file, num_samples, num_features, features_used, max_depth, num_nodes, accuracy, precision, recall, f1, model_size_kb, training_time_s, inference_ms, ram_mb, notes`

- `experiment_id`: `S5-<run>`, for example `S5-A`, `S5-A_no_port`
- `stage`: `5-baseline`
- `num_samples`: training rows + test rows
- `max_depth`: the depth the untuned tree actually grew to (not a setting)
- `precision`, `recall`, `f1`: macro averages
- `inference_ms`: milliseconds per 1,000 test rows
- `ram_mb`: empty here; Step 9b measures memory and fills it in
- `notes`: the settings, plus attack recall, missed attacks and false alarms

Also saved: the per-class recall table (`S5_per_class_recall.csv`) and a copy of
`feature_sets.json`.

In [ ]:
# Step 7 — Save the results in the same layout as experiments/results.csv
RESULTS_COLUMNS = ["experiment_id", "date", "stage", "dataset_file", "num_samples", "num_features",
                   "features_used", "max_depth", "num_nodes", "accuracy", "precision", "recall", "f1",
                   "model_size_kb", "training_time_s", "inference_ms", "ram_mb", "notes"]
RUN_ORDER = ["A", "B", "C", "D", "A_no_port", "A_no_port_no_window", "C_no_port", "D_no_port"]
today = datetime.date.today().isoformat()

rows = []
for name in RUN_ORDER:
    r = results[name]
    rows.append({
        "experiment_id": f"S5-{name}",
        "date": today,
        "stage": "5-baseline",
        "dataset_file": "Wednesday-workingHours",
        "num_samples": len(X_train) + len(X_test),
        "num_features": r["num_features"],
        "features_used": name,
        "max_depth": r["depth"],                 # the depth the tree grew to, not a setting
        "num_nodes": r["num_nodes"],
        "accuracy": round(r["accuracy"], 6),
        "precision": round(r["precision_macro"], 6),
        "recall": round(r["recall_macro"], 6),
        "f1": round(r["f1_macro"], 6),
        "model_size_kb": round(r["model_size_kb"], 2),
        "training_time_s": round(r["training_time_s"], 3),
        "inference_ms": round(r["inference_ms_per_1000"], 4),
        "ram_mb": "",                            # empty here; Step 9b measures it and fills it in
        "notes": (f"untuned DecisionTreeClassifier(random_state=42); precision/recall/f1 are macro; "
                  f"inference_ms = ms per 1000 test rows (median of {N_PREDICT_REPEATS} predict runs); "
                  f"attack recall={r['attack_recall']:.6f}; missed attacks={r['missed']}; "
                  f"false alarms={r['false_alarms']}"),
    })
results_df = pd.DataFrame(rows, columns=RESULTS_COLUMNS)

# Save the CSVs to the results folder
out_csv = RESULTS_DIR / "S5_baseline_results.csv"
results_df.to_csv(out_csv, index=False)
recall_table(RUN_ORDER).to_csv(RESULTS_DIR / "S5_per_class_recall.csv")
with open(RESULTS_DIR / "feature_sets.json", "w") as f:   # copy for the repo's experiments/ folder
    json.dump(feature_sets, f, indent=4)
print("Saved:", out_csv)
print("Saved:", RESULTS_DIR / "S5_per_class_recall.csv")
print("Saved:", RESULTS_DIR / "feature_sets.json")
display(results_df)

# The same rows in results.csv format
print("\n----- results.csv rows -----")
print(results_df.to_csv(index=False, header=False), end="")

## Step 8 — Caveats

- **The scores are likely optimistic.** One day of traffic was split at random, so
  near-identical rows from the same attack burst can land in both sets. A high score here
  does not show the tree would work on another day or network; notebook 07 tests this on
  the Friday afternoon file.
- **Rare classes are unreliable.** Heartbleed has only 2 test rows, so its recall can only
  be 0, 0.5 or 1. It is reported but no claim is built on it.
- **Attack recall matters more than accuracy.** About two thirds of the rows are BENIGN,
  so accuracy stays high even when attacks are missed.
- **The shortcut check is a result in itself.** A drop without the shortcut features shows
  how much the tree relies on how the lab was set up.

In [ ]:
# Step 8 — The numbers behind the caveats

# 1. Rare classes: how many test rows is each per-class number built on?
support = pd.Series(results["A"]["per_class_support"]).sort_values()
print("Test rows per class:")
print(support.to_string())
tiny = support[support < 50]
if len(tiny):
    print(f"-> Fewer than 50 test rows: {', '.join(tiny.index)}. "
          "One extra mistake moves their recall a lot; report them, but don't build a claim on them.")

# 2. Accuracy next to attack recall (missing a real attack is the worst mistake)
print("\nAccuracy vs attack recall:")
display(pd.DataFrame([results[n] for n in RUN_ORDER])[
    ["run", "num_features", "accuracy", "attack_recall", "missed", "false_alarms"]].round(6))

# 3. Did the shortcut check change things?
print("Shortcut check (change in macro F1 and attack recall vs the parent run):")
display(shortcut_compare[["run", "compared with", "change in macro F1", "change in attack recall",
                          "change in missed"]].round(6))

print("Reminder: this is ONE day of traffic split at random, so near-identical rows from the same "
      "attack burst can sit in both sets. A high score here does not show it would work on another day "
      "or another network.")

## Step 9 — Follow-up checks (added and first run 23-10-2026)

Two gaps found when reviewing Steps 1-8:

1. **Macro F1 is dominated by Heartbleed.** Heartbleed has only 2 test rows, but macro F1 gives it the same weight as BENIGN (83,348 rows). List C's macro F1 of 0.83 comes almost entirely from missing both Heartbleed rows; every other class is above 0.98 recall. Step 9a reports scores that are not driven by 2 rows.
2. **RAM was never measured**, even though memory is half of the research question (`ram_mb` is empty in `results.csv`). Step 9b measures it.

Run Steps 1-8 first in the same session (these cells use `results`, `predictions`, `RUN_ORDER` and `results_df`).

**Step 9a columns**
- `macro_f1_without_heartbleed`: macro F1 over the 5 classes that have enough test rows. This is the score used to compare runs.
- `weighted_f1`: F1 where each class counts in proportion to its number of rows. It is close to accuracy.
- `heartbleed_caught_of_2`: reported only. With 2 rows, 0, 1 or 2 caught is mostly luck.

**Step 9b columns**
- `tree_in_memory_mb`: the memory the tree itself takes up (its nodes and leaf values).
- `predict_1000_rows_peak_mb`: the extra memory needed while classifying 1,000 flows at once. Most of this is the batch of input data. Fewer features means less data per flow, so this part shrinks as the feature list shrinks.
- `ram_mb` in `results.csv` = the two added together.
- **What it does not include:** Python, pandas and scikit-learn themselves, which take far more memory (often 100+ MB) than the tree. So this is the memory *the model* needs, not the memory the whole program needs. A device that runs the tree without Python (for example as C code) would only need the tree part plus one flow's input.

In [ ]:
# Step 9a — Scores without Heartbleed
from sklearn.metrics import f1_score

MAIN_CLASSES = [c for c in CLASS_LABELS if c != "Heartbleed"]

rows = []
for name in RUN_ORDER:
    y_pred = predictions[name]
    rows.append({
        "run": name,
        "num_features": results[name]["num_features"],
        "macro_f1_all_classes": results[name]["f1_macro"],
        "macro_f1_without_heartbleed": f1_score(y_test, y_pred, labels=MAIN_CLASSES,
                                                average="macro", zero_division=0),
        "weighted_f1": f1_score(y_test, y_pred, average="weighted", zero_division=0),
        "attack_recall": results[name]["attack_recall"],
        "heartbleed_caught_of_2": int(((y_test == "Heartbleed").to_numpy() & (y_pred == "Heartbleed")).sum()),
    })

no_heartbleed = pd.DataFrame(rows)
display(no_heartbleed.round(4))
no_heartbleed.to_csv(RESULTS_DIR / "S5_scores_without_heartbleed.csv", index=False)
print("Saved:", RESULTS_DIR / "S5_scores_without_heartbleed.csv")

In [ ]:
# Step 9b — Memory needed to hold each tree and classify 1,000 flows
# - Tree memory: a scikit-learn tree keeps its nodes and leaf values in two C arrays
#   (allocated outside Python, so tracemalloc can't see them); we read their exact size.
# - Prediction memory: tracemalloc measures the peak extra memory Python/NumPy
#   allocate while classifying one batch of 1,000 flows.
import gc
import tracemalloc

BATCH_ROWS = 1000

ram_rows = []
for name in RUN_ORDER:
    model = joblib.load(MODEL_DIR / f"S5_baseline_{name}.joblib")
    tree_state = model.tree_.__getstate__()
    tree_mb = (tree_state["nodes"].nbytes + tree_state["values"].nbytes) / 1024**2

    batch = X_test[list(model.feature_names_in_)].iloc[:BATCH_ROWS]
    gc.collect()
    tracemalloc.start()
    model.predict(batch)
    _, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    predict_mb = peak / 1024**2

    ram_rows.append({
        "run": name,
        "tree_in_memory_mb": tree_mb,
        "predict_1000_rows_peak_mb": predict_mb,
        "total_mb": tree_mb + predict_mb,
    })
    del model, batch

ram = pd.DataFrame(ram_rows)
display(ram.round(4))

# Fill the empty ram_mb column (tree + prediction peak) and re-save the results file
results_df["ram_mb"] = results_df["features_used"].map(ram.set_index("run")["total_mb"].round(4))
results_df.to_csv(RESULTS_DIR / "S5_baseline_results.csv", index=False)
ram.to_csv(RESULTS_DIR / "S5_memory.csv", index=False)
print("Updated:", RESULTS_DIR / "S5_baseline_results.csv")
print("Saved:", RESULTS_DIR / "S5_memory.csv")
print("\n----- results.csv rows, with ram_mb -----")
print(results_df.to_csv(index=False, header=False), end="")